# 04 · Ingeniería de características con `Pipeline`

**Módulo 2 · Sesión 5** — Ingeniería de características

## Objetivos

En el notebook 01 llegamos a una lista de decisiones sobre el Titanic, pero no aplicamos
ninguna. En el 03 vimos por qué: aplicarlas fuera de un `Pipeline` invalida la evaluación.
Ahora las implementamos correctamente.

1. `ColumnTransformer`: tratamientos distintos para columnas distintas.
2. Imputación, codificación y escalado, cada uno donde corresponde.
3. **Crear** características nuevas dentro del pipeline.
4. Selección de características: filtro, envoltura y embebidos.
5. Comparar todo de forma honesta con validación cruzada.
6. Guardar el pipeline entrenado como un artefacto único.

## Paquetes

`numpy`, `pandas`, `matplotlib`, `scikit-learn`, `joblib`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-2-datos-caracteristicas/notebooks

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

SEMILLA = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

## 1. Punto de partida

Retomamos el Titanic con las decisiones del notebook 01: eliminamos `alive` (fuga), y
`class`, `embark_town` y `adult_male` (redundantes).

In [ ]:
crudo = pd.read_csv("../datos/titanic.csv")
datos = crudo.drop(columns=["alive", "class", "embark_town", "adult_male"])

y = datos["survived"]
X = datos.drop(columns=["survived"])

print(f"X: {X.shape}  ·  tasa de supervivencia: {y.mean():.1%}")
print(f"\nColumnas: {list(X.columns)}")
print(f"\nFaltantes:\n{X.isna().sum()[X.isna().sum() > 0].to_string()}")

**Lo primero de todo: apartar el conjunto de prueba.** Antes de mirar nada más.

In [ ]:
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y
)
print(f"Entrenamiento: {len(X_entrena)}  ·  Prueba: {len(X_prueba)}")
print(f"Proporción de supervivientes — entrenamiento: {y_entrena.mean():.3f}, prueba: {y_prueba.mean():.3f}")

> **`stratify=y`** mantiene la misma proporción de clases en ambos conjuntos. Con clases
> desbalanceadas es prácticamente obligatorio: sin él, una partición desafortunada puede
> dejar muy pocos casos positivos en prueba y volver la evaluación inestable.

## 2. Cada columna necesita un trato distinto

Las variables numéricas y las categóricas no se preprocesan igual:

| | Numéricas | Categóricas |
|---|---|---|
| Imputación | Mediana | Valor más frecuente |
| Transformación | Escalado | Codificación one-hot |

`ColumnTransformer` aplica un pipeline distinto a cada grupo de columnas y concatena el
resultado.

In [ ]:
numericas = ["age", "sibsp", "parch", "fare"]
categoricas = ["pclass", "sex", "embarked", "who", "alone"]

flujo_numerico = Pipeline(
    [
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ]
)

flujo_categorico = Pipeline(
    [
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ]
)

preprocesador = ColumnTransformer(
    [
        ("num", flujo_numerico, numericas),
        ("cat", flujo_categorico, categoricas),
    ],
    remainder="drop",       # descarta lo no listado (aquí, 'deck')
)

X_transformado = preprocesador.fit_transform(X_entrena)
print(f"De {X_entrena.shape[1]} columnas a {X_transformado.shape[1]} tras el preprocesamiento")
print(f"\nNombres generados:\n{list(preprocesador.get_feature_names_out())}")

Detalles que importan:

- **`handle_unknown="ignore"`**: si en producción aparece una categoría que no estaba en
  entrenamiento, el codificador la deja en ceros en lugar de fallar. Sin esto, tu API se cae
  el día que llegue un valor nuevo.
- **`drop="first"`** elimina una categoría por variable para evitar colinealidad perfecta
  entre las columnas generadas. Importa en modelos lineales (sesión 7); es indiferente en
  árboles.
- **`remainder="drop"`** descarta explícitamente lo que no se listó. Es preferible a
  `"passthrough"`: obliga a decidir sobre cada columna en lugar de dejar pasar sorpresas.

## 3. Crear características nuevas

El EDA dejó tres ideas pendientes: combinar `sibsp` y `parch`, aprovechar la ausencia de
`deck`, y transformar `fare` con logaritmo.

In [ ]:
def construir_caracteristicas(df):
    """Crea variables nuevas a partir de las originales. Sin aprender nada de los datos."""
    df = df.copy()
    df["tamano_familia"] = df["sibsp"] + df["parch"] + 1
    df["tiene_camarote"] = df["deck"].notna().astype(int)
    df["falta_edad"] = df["age"].isna().astype(int)
    df["log_fare"] = np.log1p(df["fare"])
    return df


ejemplo = construir_caracteristicas(X_entrena)
print(ejemplo[["sibsp", "parch", "tamano_familia", "tiene_camarote", "falta_edad", "fare", "log_fare"]].head().to_string())

> **Por qué esta función es segura.** Cada variable nueva se calcula **fila a fila**, sin
> usar ningún estadístico del conjunto de datos. No hay medias, ni medianas, ni conteos
> globales. Por eso puede aplicarse antes de partir sin causar fuga.
>
> La regla general: una transformación que solo mira la propia fila es segura; una que
> necesita mirar el resto del dataset debe ir dentro del `Pipeline`. Si dudas, mételo en el
> `Pipeline`.

Ahora sí, el pipeline completo: primero las características nuevas, después el
preprocesamiento por tipo de columna.

In [ ]:
numericas_ext = numericas + ["tamano_familia", "log_fare"]
categoricas_ext = categoricas + ["tiene_camarote", "falta_edad"]

preprocesador_ext = ColumnTransformer(
    [
        ("num", flujo_numerico, numericas_ext),
        ("cat", flujo_categorico, categoricas_ext),
    ],
    remainder="drop",
)

flujo_completo = Pipeline(
    [
        ("caracteristicas", FunctionTransformer(construir_caracteristicas)),
        ("preprocesar", preprocesador_ext),
        ("modelo", LogisticRegression(max_iter=1000, random_state=SEMILLA)),
    ]
)

flujo_completo.fit(X_entrena, y_entrena)
print("Pipeline entrenado.")
print(f"Accuracy en prueba: {flujo_completo.score(X_prueba, y_prueba):.4f}")

Un solo objeto contiene la creación de variables, la imputación, el escalado, la
codificación y el modelo. `fit` aprende todo con el entrenamiento; `predict` solo aplica.

## 4. ¿Aportaron algo las variables nuevas?

La pregunta no se responde con una corazonada, sino comparando con validación cruzada.

In [ ]:
flujo_base = Pipeline(
    [
        ("preprocesar", preprocesador),
        ("modelo", LogisticRegression(max_iter=1000, random_state=SEMILLA)),
    ]
)

resultados = {}
for nombre, flujo in [("Solo originales", flujo_base), ("Con variables nuevas", flujo_completo)]:
    puntajes = cross_val_score(flujo, X_entrena, y_entrena, cv=cv, scoring="accuracy")
    resultados[nombre] = puntajes
    print(f"{nombre:22s} accuracy = {puntajes.mean():.4f} (±{puntajes.std():.4f})")

diferencia = resultados["Con variables nuevas"].mean() - resultados["Solo originales"].mean()
print(f"\nDiferencia: {diferencia:+.4f}")

**Las variables nuevas no mejoraron nada.** De hecho la diferencia sale ligeramente
negativa, y la desviación entre pliegues es cinco veces mayor que la diferencia: lo único
que se puede concluir es que **no hay diferencia detectable**.

Conviene detenerse aquí, porque es la parte que los tutoriales suelen ocultar. La ingeniería
de características tiene fama de ser donde se gana la partida, y a veces lo es — pero:

- `tamano_familia` es una combinación lineal de `sibsp` y `parch`, que el modelo ya tenía.
  Una regresión logística puede construirla sola.
- `log_fare` es una transformación monótona de `fare`, que también estaba.
- `falta_edad` y `tiene_camarote` sí aportan información nueva, pero poca frente a lo que ya
  explican el sexo y la clase.

> **La lección.** Crear variables es barato; **demostrar que ayudan es lo caro**. Una
> diferencia menor que la variabilidad entre pliegues no es una mejora, es ruido. Reportarla
> como mejora es una forma sutil de engañarse.

La sesión 8 da las herramientas para decidir en casos así: validación cruzada repetida e
intervalos de confianza sobre la diferencia.

## 5. Selección de características

Tenemos ahora más variables que al empezar. ¿Sobran algunas?

Los tres enfoques clásicos:

| Método | Cómo decide | Coste | Ejemplo |
|---|---|---|---|
| **Filtro** | Estadístico por variable, ignora el modelo | Bajo | `SelectKBest` |
| **Envoltura** (*wrapper*) | Entrena el modelo repetidamente | Alto | `RFE` |
| **Embebido** | La selección es parte del entrenamiento | Medio | Lasso, importancia de árboles |

**Todos van dentro del `Pipeline`**, por lo que vimos en el notebook 03.

In [ ]:
def flujo_con_seleccion(selector):
    return Pipeline(
        [
            ("caracteristicas", FunctionTransformer(construir_caracteristicas)),
            ("preprocesar", preprocesador_ext),
            ("seleccionar", selector),
            ("modelo", LogisticRegression(max_iter=1000, random_state=SEMILLA)),
        ]
    )


comparacion = {"Sin selección (todas)": flujo_completo}
for k in [5, 8, 12]:
    comparacion[f"Filtro: mejores {k}"] = flujo_con_seleccion(SelectKBest(f_classif, k=k))
comparacion["Envoltura: RFE a 8"] = flujo_con_seleccion(
    RFE(LogisticRegression(max_iter=1000, random_state=SEMILLA), n_features_to_select=8)
)

print(f"{'Estrategia':26s} {'accuracy':>10} {'desv.':>8}")
print("-" * 46)
tabla = {}
for nombre, flujo in comparacion.items():
    puntajes = cross_val_score(flujo, X_entrena, y_entrena, cv=cv, scoring="accuracy")
    tabla[nombre] = puntajes
    print(f"{nombre:26s} {puntajes.mean():>10.4f} {puntajes.std():>8.4f}")

Ninguna estrategia de selección mejora de forma clara. Con ~15 variables y ~700
observaciones **no hay problema de dimensionalidad que resolver**: la selección de
características es una herramienta para cuando hay cientos o miles de variables, no para
aplicarla por rutina.

Lo que sí aporta es **simplicidad**: si con 8 variables se obtiene lo mismo que con 15, el
modelo más pequeño es preferible por interpretabilidad, coste y mantenimiento.

### Método embebido: importancia según un modelo de árboles

In [ ]:
flujo_bosque = Pipeline(
    [
        ("caracteristicas", FunctionTransformer(construir_caracteristicas)),
        ("preprocesar", preprocesador_ext),
        ("modelo", RandomForestClassifier(n_estimators=300, random_state=SEMILLA)),
    ]
)
flujo_bosque.fit(X_entrena, y_entrena)

nombres = flujo_bosque.named_steps["preprocesar"].get_feature_names_out()
importancias = pd.Series(
    flujo_bosque.named_steps["modelo"].feature_importances_, index=nombres
).sort_values(ascending=False)

print(importancias.round(4).to_string())

In [ ]:
fig, eje = plt.subplots(figsize=(8, 5))
importancias.head(12).iloc[::-1].plot(kind="barh", ax=eje)
eje.set_xlabel("Importancia")
eje.set_title("Importancia de variables según Random Forest")
plt.tight_layout()
plt.show()

A primera vista el resultado **contradice** al EDA: arriba aparece `age`, cuando sabemos que
el sexo era con diferencia el factor más determinante (74 % de supervivencia frente a 19 %).

No es que el bosque se equivoque: es que esta gráfica engaña de dos formas a la vez, y las
dos son visibles aquí.

1. **El crédito se reparte entre variables que dicen lo mismo.** El sexo está codificado tres
   veces: `sex_male`, `who_man` y `who_woman`. Sumadas dan ≈ 0.30, muy por encima de `age`.
   Lo mismo pasa con `fare` y `log_fare`, que juntas suman ≈ 0.28. Ninguna columna individual
   refleja la importancia real de su variable.
2. **La importancia por impureza favorece a las variables con muchos valores distintos.**
   `age` es continua y ofrece cientos de puntos de corte posibles; una binaria como
   `sex_male` ofrece uno. El árbol usa `age` muchas más veces, aunque cada uso aporte menos.

> **Cómo leer esto.** La importancia por impureza sirve como orientación rápida, no como
> verdad. Cuando haya variables correlacionadas o de cardinalidad muy distinta —es decir,
> casi siempre— hay que usar *permutation importance* o SHAP, que veremos en la sesión 11.
>
> Y una consecuencia práctica inmediata: **no uses esta gráfica para descartar variables**.
> Podrías eliminar `sex_male` viendo su 0.10, sin darte cuenta de que el sexo es lo más
> informativo del dataset.

## 6. Evaluación final

Ninguna de las variantes probadas resultó mejor que otra de forma medible. Cuando eso pasa,
el criterio de desempate es la **parsimonia**: en un proyecto real elegiríamos el pipeline
más simple, `flujo_base`, porque cuesta menos mantener y explicar.

Aquí nos quedamos con `flujo_completo` por una razón distinta y que conviene declarar: es el
que incluye la creación de variables, y queremos desplegar ese pipeline completo en la
sección 7. Es una decisión pedagógica, no estadística.

Solo ahora, con el modelo ya elegido, tocamos el conjunto de prueba. **Una sola vez.**

In [ ]:
flujo_final = flujo_completo
flujo_final.fit(X_entrena, y_entrena)

acc_entrena = flujo_final.score(X_entrena, y_entrena)
acc_prueba = flujo_final.score(X_prueba, y_prueba)
cv_media = cross_val_score(flujo_final, X_entrena, y_entrena, cv=cv).mean()

referencia = y_prueba.value_counts(normalize=True).max()

print(f"Accuracy en entrenamiento:            {acc_entrena:.4f}")
print(f"Accuracy en validación cruzada:       {cv_media:.4f}")
print(f"Accuracy en prueba (una sola vez):    {acc_prueba:.4f}")
print(f"Referencia trivial (clase mayoritaria): {referencia:.4f}")
print(f"\nMejora sobre la referencia: {acc_prueba - referencia:+.4f}")

Las tres cifras son coherentes entre sí, lo que sugiere que no hay sobreajuste grave, y el
modelo supera con holgura a la referencia trivial.

> La accuracy es una métrica pobre para clasificación, sobre todo con clases desbalanceadas.
> La usamos aquí porque el foco del módulo son los datos, no el modelo. En la sesión 9
> haremos esta evaluación en serio: matriz de confusión, precisión, recall, ROC-AUC y ajuste
> del umbral.

## 7. Guardar el pipeline

El artefacto que se despliega **no es el modelo**: es el pipeline completo. Contiene las
medianas de imputación, las medias y desviaciones del escalado, las categorías del
codificador y los coeficientes. Sin eso, en producción no se puede reproducir el
preprocesamiento.

In [ ]:
import tempfile
from pathlib import Path

destino = Path(tempfile.mkdtemp()) / "pipeline-titanic.joblib"
joblib.dump(flujo_final, destino)
print(f"Guardado: {destino.name} ({destino.stat().st_size / 1024:.0f} KB)")

cargado = joblib.load(destino)

# Un pasajero nuevo, con la misma estructura que los datos originales.
pasajero = pd.DataFrame(
    [{"pclass": 3, "sex": "female", "age": 22.0, "sibsp": 1, "parch": 0,
      "fare": 7.25, "embarked": "S", "who": "woman", "deck": np.nan, "alone": False}]
)
probabilidad = cargado.predict_proba(pasajero)[0, 1]
print(f"\nProbabilidad de supervivencia del pasajero de ejemplo: {probabilidad:.1%}")
print(f"¿Predicciones idénticas tras recargar? "
      f"{np.array_equal(flujo_final.predict(X_prueba), cargado.predict(X_prueba))}")

destino.parent.rmdir() if not any(destino.parent.iterdir()) else destino.unlink()

Fíjate en que el pasajero nuevo se pasa **crudo**, con `NaN` en `deck` incluido. El pipeline
se encarga de todo: crear variables, imputar, escalar, codificar y predecir. Ese es
exactamente el objeto que expondremos como API en la sesión 14.

## Resumen

| Herramienta | Para qué |
|---|---|
| `Pipeline` | Encadena pasos y garantiza que solo aprendan del entrenamiento |
| `ColumnTransformer` | Tratamiento distinto para numéricas y categóricas |
| `SimpleImputer` | Rellena faltantes con un estadístico aprendido del entrenamiento |
| `StandardScaler` | Media 0 y desviación 1 |
| `OneHotEncoder` | Categorías a columnas binarias (`handle_unknown="ignore"`) |
| `FunctionTransformer` | Mete tus propias transformaciones fila a fila en el pipeline |
| `SelectKBest` / `RFE` | Selección por filtro y por envoltura |
| `joblib` | Guarda el pipeline completo como un artefacto |

**Las tres ideas que hay que llevarse:**

1. El pipeline es el modelo. No se despliega un clasificador, se despliega el pipeline.
2. Crear variables es barato; demostrar que ayudan requiere validación cruzada, y a menudo
   la respuesta honesta es "no se puede afirmar".
3. La selección de características resuelve un problema de dimensionalidad. Si no lo tienes,
   no la necesitas.

## Para practicar

1. Añade una variable `es_madre` (mujer, mayor de 18, con `parch > 0`). ¿Mejora la
   validación cruzada? ¿La diferencia supera a la desviación entre pliegues?
2. Cambia la imputación de `age` de la mediana global a la mediana **por clase**. Necesitarás
   un transformador propio. ¿Mejora algo? El notebook 01 predecía que sí — compruébalo.
3. Sustituye la regresión logística por `RandomForestClassifier` sin cambiar nada más. ¿Sigue
   haciendo falta el escalado? ¿Por qué?
4. Quita `handle_unknown="ignore"` e intenta predecir con un pasajero cuyo `embarked` sea
   `"X"`. Observa el error: es exactamente el que tumbaría tu API en producción.